###Usecase2: Telecom data - FC, CDC, Watermarking/Checkpointing, CDF, SCD

![](devices_cdc1.png)

**Prerequisites - One time activity (done by admin, based on the request)**

STEP 1: CREATE CONNECTION (FOREIGN CATALOG)

In [0]:
%sql
CREATE CONNECTION telecom_gcp_mysql_conn3
TYPE MYSQL
OPTIONS (
  host '35.223.80.16',
  port '3306',
  user 'inceptez',
  password 'Inceptez@123');--we can better store this password in a secret scope/keyvault

STEP 2: CREATE FOREIGN CATALOG

In [0]:
%sql
CREATE FOREIGN CATALOG telecom_federated_we47_fc
USING CONNECTION telecom_gcp_mysql_conn3;

STEP 3: VERIFY SOURCE TABLE (LIVE QUERY)

In [0]:
%sql
SELECT * FROM telecom_federated_we47_fc.telecom_devices.device_master;

In [0]:
_sqldf.show()

STEP 4: CREATE LAYERS (BRONZE / SILVER / GOLD)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS telecom_bronze;
CREATE SCHEMA IF NOT EXISTS telecom_silver;
CREATE SCHEMA IF NOT EXISTS telecom_gold;

**Interview Questions wrt CDC, CDF, SCD1 & SCD2**

Change Data Feed (CDF)

Question: Why use Change Data Feed (CDF)?

Answer: It saves time and compute costs by only processing the rows that actually changed, instead of scanning the entire table.

Question: What does CDF output when a row is updated?

Answer: Two rows: the old version (pre-image) and the new version (post-image).

Slowly Changing Dimension Type 1 (SCD1)

Question: What is SCD Type 1?

Answer: It stores only the latest data. When a change happens, the old data is simply overwritten and lost.

Question: When should you use SCD Type 1?

Answer: When you only care about the current status and don't need to track historical changes (e.g., fixing a typo).

Slowly Changing Dimension Type 2 (SCD2)

Question: What is SCD Type 2?

Answer: It keeps the full history of changes. Instead of overwriting, it adds a new row for every update and uses tracking columns (like start/end dates) to show the timeline.

Question: How do you handle a "delete" in SCD Type 2?

Answer: You don't actually delete the row. You do a "soft delete" by marking the record as inactive and stamping an end date.

In [0]:
%sql
select * from telecom_federated_we47_fc.telecom_devices.device_master

In [0]:
%sql
CREATE TABLE IF NOT EXISTS telecom_bronze.bronze_device_master_6 (
  device_id INT, device_type STRING, brand STRING, model STRING, os STRING, 
  owner_customer_id INT, status STRING, updated_at TIMESTAMP);
--1. CDC (History load for the first time alone) applying Watermarking Feature using Foreign Catalog
INSERT INTO telecom_bronze.bronze_device_master_6
SELECT 
    device_id, 
    device_type, 
    brand, 
    model, 
    os, 
    owner_customer_id, 
    status, 
    updated_at
    --,    current_timestamp() AS ingestion_time
FROM telecom_federated_we47_fc.telecom_devices.device_master
WHERE updated_at > (SELECT COALESCE(MAX(updated_at), '1900-01-01') 
    FROM telecom_bronze.bronze_device_master_6);--This is checkpointing or watermarking feature to achieve incremental load/CDC

--delete from telecom_bronze.bronze_device_master_6 
--where device_id not in (select distinct device_id from telecom_federated_we47_fc.telecom_devices.device_master);

**Verification Queries**

In [0]:
%sql
select * from telecom_bronze.bronze_device_master_6

In [0]:
%sql
SELECT * FROM telecom_silver.device_master_silver_6;

In [0]:
%sql
select * from telecom_gold.device_master_gold_scd1_6

In [0]:
%sql
select * from telecom_gold.device_master_gold_scd2_6